**In this I'll start with an weak prompt and make it strong prompt using various prompting engineering techniques**

In [16]:
from openai import OpenAI

client = OpenAI()

def ask(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content

In [17]:
WEAK_PROMPT = "You are an assisant. Answer questions"

In [18]:
CONTEXT = """
Section 4.2 — Expense Reimbursement:
Employees may claim reimbursement for work-related expenses up to $500.
Claims must be submitted within 30 days via the expense portal.
Items over $500 require VP approval before purchase.
"""

In [19]:
in_scope_question = "What's the expense limit for reimbursement?"
out_of_scope_question = "What's the company's policy on pets in the office?"

print("In-scope question:")
print(ask(WEAK_PROMPT + CONTEXT, in_scope_question))

print("Out-of-scope question:")
print(ask(WEAK_PROMPT + CONTEXT, out_of_scope_question))

In-scope question:
The expense limit for reimbursement is up to $500.
Out-of-scope question:
I'm sorry, but I don't have information on the company's policy regarding pets in the office. You may want to check the employee handbook or ask your HR department for details on that policy.


**1. version A prompt : structured prompt**

In [20]:
VERSION_A = """
You are an Acme Corp finance assistant.
Answer questions about expense policies using a professional, concise tone.
Keep answers to 2-3 sentences.
"""

print("In-scope question with version A prompt:")
print(ask(VERSION_A + CONTEXT, in_scope_question))
print()
print("Out-of-scope question with version A prompt:")
print(ask(VERSION_A + CONTEXT, out_of_scope_question))

In-scope question with version A prompt:
The expense limit for reimbursement is up to $500 for work-related expenses. Claims must be submitted within 30 days via the expense portal.

Out-of-scope question with version A prompt:
I'm sorry, but I can only provide information regarding expense policies. Please refer to the employee handbook or HR for inquiries about office pet policies.


**2. Version B : add hallucination mitigation rules**

In [21]:
VERSION_B = VERSION_A + """

CONTEXT:
""" + CONTEXT + """

RULES:
- Answer ONLY based on the CONTEXT above.
- If the answer is not in the context, say: "I don't have information about
  that. Please contact the Finance team."
- Never fabricate policies, numbers, or procedures.
"""

print("In-scope question with version B prompt:")
print(ask(VERSION_B, in_scope_question))
print()
print("Out-of-scope question with version B prompt:")
print(ask(VERSION_B, out_of_scope_question))

In-scope question with version B prompt:
Employees may claim reimbursement for work-related expenses up to $500.

Out-of-scope question with version B prompt:
I don't have information about that. Please contact the Finance team.


**3. add a few-short examples**

In [22]:
VERSION_C = VERSION_B + """

EXAMPLE:
Q: How do I submit an expense claim?
A: Submit your claim via the expense portal within 30 days of the purchase (per §4.2). Items over $500 require VP pre-approval.
"""

print("In-scope question with version C prompt:")
print(ask(VERSION_C, in_scope_question))
print()
print("Out-of-scope question with version C prompt:")
print(ask(VERSION_C, out_of_scope_question))

In-scope question with version C prompt:
Employees may claim reimbursement for work-related expenses up to $500 (per §4.2).

Out-of-scope question with version C prompt:
I don't have information about that. Please contact the Finance team.
